# Model Training Notebook
**AAI-521 Computer Vision Project**

This notebook trains three different models for binary classification of AI-generated vs Real images:
1. **Custom CNN** - A lightweight 3-layer convolutional neural network
2. **ResNet50** - Transfer learning with frozen backbone
3. **EfficientNet-B2** - Transfer learning with advanced architecture

All models are trained using mixed precision (AMP) for efficient GPU utilization and saved to the `../models/` directory.

## Prerequisites

**Important:** Before running this notebook, you must first complete `01_data_preparation.ipynb` to:
- Download and prepare the dataset
- Create train/val/test splits
- Generate the required data loaders

This notebook assumes the data loaders are saved and ready to load.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models
import os
import time
import copy

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)
print("Models directory ready")

# Note: This assumes train_loader, val_loader, and test_loader 
# were created and saved in 01_data_preparation.ipynb
# For this notebook, we'll need to recreate them or load them
# (In production, you would load the saved data loaders here)

---
# SECTION 1: Custom CNN Model

We'll start with a lightweight custom CNN architecture featuring:
- 3 convolutional layers with batch normalization
- Max pooling and dropout for regularization
- Binary classification output (1 class with sigmoid)

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=1):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate flattened features size dynamically
        # Input image size is 224x224. After 3 pooling layers (each 2x2), size becomes 224/8 = 28
        # So, the size before the first linear layer is 128 * 28 * 28
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(-1, 128 * 28 * 28)  # Flatten the tensor
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Instantiate the CNN model
cnn_model = CNN(num_classes=1).to(device)

print("CNN Architecture:")
print(cnn_model)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

In [ ]:
# CNN Training Setup
criterion_cnn = nn.BCEWithLogitsLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.001)
scaler_cnn = torch.cuda.amp.GradScaler()

# Training functions
def train_epoch(model, dataloader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        # Forward pass with Automatic Mixed Precision
        with torch.amp.autocast(device_type='cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Backward pass with scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        # Calculate accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy


def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

print("CNN training setup complete with AMP support")

In [ ]:
# CNN Training Loop
num_epochs_cnn = 10
best_cnn_val_acc = 0.0
cnn_model_path = '../models/cnn_best.pth'

print(f"Training CNN for {num_epochs_cnn} epochs...")
print("=" * 60)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

start_time = time.time()

for epoch in range(num_epochs_cnn):
    # Train for one epoch
    train_loss, train_acc = train_epoch(
        cnn_model, train_loader, criterion_cnn, optimizer_cnn, device, scaler_cnn
    )
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validate
    val_loss, val_acc = evaluate_model(cnn_model, val_loader, criterion_cnn, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs_cnn}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_cnn_val_acc:
        best_cnn_val_acc = val_acc
        torch.save(cnn_model.state_dict(), cnn_model_path)
        print(f"  💾 Saved new best model (Val Acc: {val_acc:.4f})")
    print()

training_time = time.time() - start_time
print(f"CNN training complete in {training_time:.2f} seconds")
print(f"Best validation accuracy: {best_cnn_val_acc:.4f}")

# Load best weights
cnn_model.load_state_dict(torch.load(cnn_model_path))
print(f"✓ Best model loaded from {cnn_model_path}")

In [ ]:
# CNN Test Evaluation
test_loss, test_acc = evaluate_model(cnn_model, test_loader, criterion_cnn, device)

print("=" * 60)
print("CNN FINAL RESULTS")
print("=" * 60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Model saved at: {cnn_model_path}")
print("=" * 60)

---
# SECTION 2: ResNet50 Transfer Learning

ResNet50 is a deep residual network pre-trained on ImageNet. We'll:
- Load the pretrained model
- Freeze all backbone layers
- Replace the final fully connected layer for binary classification
- Train only the new FC layer

In [ ]:
# Load pretrained ResNet50
print("Loading pretrained ResNet50...")
try:
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
except:
    resnet = models.resnet50(pretrained=True)

# Freeze all layers in the backbone
for param in resnet.parameters():
    param.requires_grad = False

# Replace the final classification layer
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 2)  # 2 classes: Real vs AI-Generated

# Move to GPU
resnet = resnet.to(device)

print("\nResNet50 Architecture (modified):")
print(f"Original features: {num_features}")
print(f"New final layer: {resnet.fc}")
print(f"Trainable parameters: {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in resnet.parameters()):,}")

In [ ]:
# ResNet50 Training Setup
criterion_resnet = nn.CrossEntropyLoss()

# Optimizer - only train the final FC layer
optimizer_resnet = optim.Adam(
    resnet.fc.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# AMP scaler
scaler_resnet = torch.cuda.amp.GradScaler()

num_epochs_resnet = 5

print("ResNet50 training configuration:")
print(f"  Loss function: {criterion_resnet}")
print(f"  Optimizer: Adam (lr=1e-4, weight_decay=1e-4)")
print(f"  Epochs: {num_epochs_resnet}")
print(f"  AMP enabled: Yes")

In [ ]:
# ResNet50 Training Loop with AMP
def train_one_epoch_amp(model, dataloader, criterion, optimizer, scaler, device):
    """Training loop with Automatic Mixed Precision"""
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Forward pass (mixed precision)
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

        # Backward pass (scaled for AMP)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update statistics
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_corrects += torch.sum(preds == labels).item()
        total += batch_size

    return running_loss / total, running_corrects / total


def evaluate_resnet(model, dataloader, criterion, device):
    """Evaluation loop"""
    model.eval()
    total_loss = 0
    total_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += torch.sum(preds == labels).item()
            total += batch_size

    return total_loss / total, total_correct / total


# Training loop
best_resnet_wts = copy.deepcopy(resnet.state_dict())
best_resnet_acc = 0.0
resnet_model_path = '../models/resnet50_best.pth'

print(f"Training ResNet50 for {num_epochs_resnet} epochs with AMP...")
print("=" * 60)

start = time.time()

for epoch in range(num_epochs_resnet):
    print(f"\nEpoch {epoch+1}/{num_epochs_resnet}")
    print("-" * 40)

    train_loss, train_acc = train_one_epoch_amp(
        resnet, train_loader, criterion_resnet, optimizer_resnet, scaler_resnet, device
    )
    val_loss, val_acc = evaluate_resnet(resnet, val_loader, criterion_resnet, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_resnet_acc:
        best_resnet_acc = val_acc
        best_resnet_wts = copy.deepcopy(resnet.state_dict())
        torch.save(best_resnet_wts, resnet_model_path)
        print(f"💾 Saved new BEST model → {resnet_model_path}")

end = time.time()

print("\n" + "=" * 60)
print("ResNet50 training complete!")
print(f"Best validation accuracy: {best_resnet_acc:.4f}")
print(f"Total training time: {end - start:.2f} seconds")
print("=" * 60)

# Load best weights back into the model
resnet.load_state_dict(best_resnet_wts)
print(f"✓ Best weights reloaded from {resnet_model_path}")

In [ ]:
# ResNet50 Test Evaluation
test_loss_resnet, test_acc_resnet = evaluate_resnet(resnet, test_loader, criterion_resnet, device)

print("=" * 60)
print("RESNET50 FINAL RESULTS")
print("=" * 60)
print(f"Test Loss: {test_loss_resnet:.4f}")
print(f"Test Accuracy: {test_acc_resnet:.4f} ({test_acc_resnet*100:.2f}%)")
print(f"Model saved at: {resnet_model_path}")
print("=" * 60)

---
# SECTION 3: EfficientNet-B2 Transfer Learning

EfficientNet-B2 is a compound-scaled network that balances depth, width, and resolution. It excels at capturing fine textures and high-frequency details in images. We'll:
- Load pretrained EfficientNet-B2
- Freeze all layers except the classifier
- Train with learning rate scheduling
- Use AMP for efficiency

In [ ]:
# Load pretrained EfficientNet-B2
print("Loading pretrained EfficientNet-B2...")
try:
    efficientnet = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
except:
    efficientnet = models.efficientnet_b2(pretrained=True)

# Freeze all layers except the classifier head
for param in efficientnet.parameters():
    param.requires_grad = False

# Replace the classifier head
# B2 final feature size = 1408
# 2 classes (Real vs AI)
num_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(num_features, 2)

# Move to GPU
efficientnet = efficientnet.to(device)

# Print summary
print("\nEfficientNet-B2 Architecture (modified):")
print(f"Original features: {num_features}")
print(f"New classifier head: {efficientnet.classifier[1]}")
print(f"Trainable parameters: {sum(p.numel() for p in efficientnet.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in efficientnet.parameters()):,}")
print("✓ EfficientNet-B2 ready for training!")

In [ ]:
# EfficientNet-B2 Training with AMP and Scheduler
print("🚀 Starting EfficientNet-B2 Training...")

# Loss function
criterion_eff = nn.CrossEntropyLoss()

# Optimizer - only trains the classifier head
optimizer_eff = optim.Adam(
    efficientnet.classifier[1].parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Learning rate scheduler
scheduler_eff = optim.lr_scheduler.StepLR(
    optimizer_eff,
    step_size=3,
    gamma=0.1
)

# AMP scaler
scaler_eff = torch.cuda.amp.GradScaler()

# Track best model
best_eff_wts = copy.deepcopy(efficientnet.state_dict())
best_eff_val_acc = 0.0
efficientnet_model_path = '../models/efficientnet_b2_best.pth'

# Number of epochs
num_epochs_eff = 5

print("EfficientNet-B2 training configuration:")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Optimizer: Adam (lr=1e-4, weight_decay=1e-4)")
print(f"  Scheduler: StepLR (step_size=3, gamma=0.1)")
print(f"  Epochs: {num_epochs_eff}")
print(f"  AMP enabled: Yes")
print("=" * 60)

start_time = time.time()

for epoch in range(num_epochs_eff):
    print(f"\nEpoch {epoch+1}/{num_epochs_eff}")
    print("-" * 40)

    # -------------------------------
    # TRAINING LOOP
    # -------------------------------
    efficientnet.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_eff.zero_grad()

        # Mixed precision forward pass
        with torch.amp.autocast(device_type="cuda"):
            outputs = efficientnet(images)
            loss = criterion_eff(outputs, labels)

        _, preds = torch.max(outputs, 1)
        batch_size = labels.size(0)

        # Mixed precision backward pass
        scaler_eff.scale(loss).backward()
        scaler_eff.step(optimizer_eff)
        scaler_eff.update()

        running_loss += loss.item() * batch_size
        running_corrects += torch.sum(preds == labels).item()
        total_samples += batch_size

    train_loss = running_loss / total_samples
    train_acc = running_corrects / total_samples

    # -------------------------------
    # VALIDATION LOOP
    # -------------------------------
    efficientnet.eval()
    val_loss = 0.0
    val_corrects = 0
    val_samples = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type="cuda"):
                outputs = efficientnet(images)
                loss = criterion_eff(outputs, labels)

            _, preds = torch.max(outputs, 1)

            val_loss += loss.item() * labels.size(0)
            val_corrects += torch.sum(preds == labels).item()
            val_samples += labels.size(0)

    val_loss /= val_samples
    val_acc = val_corrects / val_samples

    scheduler_eff.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # Save best EfficientNet model
    if val_acc > best_eff_val_acc:
        best_eff_val_acc = val_acc
        best_eff_wts = copy.deepcopy(efficientnet.state_dict())
        print(f"💾 Saved new BEST EfficientNet model → {efficientnet_model_path}")
        torch.save(best_eff_wts, efficientnet_model_path)

# Load best weights back into model
efficientnet.load_state_dict(best_eff_wts)

total_time = time.time() - start_time

print("\n" + "=" * 60)
print("🎉 EfficientNet Training Complete!")
print(f"Best Validation Accuracy: {best_eff_val_acc:.4f}")
print(f"Total training time: {total_time:.2f} seconds")
print("=" * 60)

In [ ]:
# EfficientNet-B2 Test Evaluation
print("🔍 Running EfficientNet Test Evaluation...")

efficientnet.eval()
test_loss_eff = 0.0
test_corrects_eff = 0
test_samples_eff = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Use AMP for faster test-time inference
        with torch.amp.autocast(device_type="cuda"):
            outputs = efficientnet(images)
            loss = criterion_eff(outputs, labels)

        _, preds = torch.max(outputs, 1)

        test_loss_eff += loss.item() * labels.size(0)
        test_corrects_eff += torch.sum(preds == labels).item()
        test_samples_eff += labels.size(0)

test_loss_eff /= test_samples_eff
test_acc_eff = test_corrects_eff / test_samples_eff

print("\n" + "=" * 60)
print("EFFICIENTNET-B2 FINAL RESULTS")
print("=" * 60)
print(f"Test Loss: {test_loss_eff:.4f}")
print(f"Test Accuracy: {test_acc_eff:.4f} ({test_acc_eff*100:.2f}%)")
print(f"Model saved at: {efficientnet_model_path}")
print("=" * 60)

---
# Training Summary

## Models Trained

All three models have been successfully trained and saved:

### 1. Custom CNN
- **Architecture**: 3-layer convolutional network with batch normalization and dropout
- **Training**: 10 epochs with BCEWithLogitsLoss (binary classification)
- **Saved Model**: `../models/cnn_best.pth`
- **Use Case**: Lightweight baseline model

### 2. ResNet50 (Transfer Learning)
- **Architecture**: Pretrained ResNet50 with frozen backbone
- **Training**: 5 epochs, only FC layer trained
- **Saved Model**: `../models/resnet50_best.pth`
- **Use Case**: Robust feature extraction, good for varied image quality

### 3. EfficientNet-B2 (Transfer Learning)
- **Architecture**: Pretrained EfficientNet-B2 with frozen backbone
- **Training**: 5 epochs with learning rate scheduling
- **Saved Model**: `../models/efficientnet_b2_best.pth`
- **Use Case**: Captures fine textures and high-frequency details

## Key Features
- ✅ All models use **Automatic Mixed Precision (AMP)** for efficient training
- ✅ Best models saved based on validation accuracy
- ✅ Binary classification: Real (0) vs AI-Generated (1)
- ✅ Models saved in standard PyTorch format (`.pth`)

## Next Steps
1. **Model Evaluation**: Run detailed evaluation with confusion matrices and ROC curves
2. **Model Comparison**: Compare all three models on the test set
3. **Inference Pipeline**: Create a script for making predictions on new images
4. **Model Deployment**: Package the best model for production use

## Notes
- The CNN uses sigmoid activation (BCEWithLogitsLoss), while ResNet50 and EfficientNet-B2 use softmax (CrossEntropyLoss)
- ResNet50 typically performs well on datasets with varied image quality
- EfficientNet-B2 excels on high-resolution, photo-realistic images
- All models are saved with state_dict only (weights), not the full architecture